# A/A Test: V2 Customer Lookalike (with city_type)

**Status: STUB — not yet runnable.**

V2 extends V3 by adding `city_type` as a matching condition and tracking
GMV metrics. Before this notebook can run, the `city_type` sourcing must
be implemented.

## What needs to be resolved

1. **Where does `city_type` come from?** The production V2 script reads it
   from `customer_lookalike_evaluation_audience_citytype`, but for A/A we
   need to derive it independently. Options:
   - `fact_segmentation_scv_key.city` mapped to `dim_reported_city.city_tier`
   - Customer's most recent order city from `dim_restaurant`

2. **What are the `city_type` values?** The production table uses this column
   but the distinct values are unknown. Check with:
   ```sql
   SELECT DISTINCT city_type FROM
   `just-data-warehouse.customer_intelligence.customer_lookalike_evaluation_audience_citytype`
   ```

3. **V2 matching conditions include `L365D_AOV_cat`** in both exact match
   and KNN — this is computed via NTILE(3) so it's already in the feature query.

## Differences from V3

| Feature | V3 | V2 |
|---------|----|----|
| `city_type` matching | No | Yes |
| `L365D_AOV_cat` matching | No | Yes |
| `L30D_promo_orders` filter | Yes | No |
| `L730D_orders` outlier filter | Yes | No |
| GMV metrics | No | Yes (`L30D_GMV`, `campaign_period_food_total`, `campaign_period_discount`) |

Once `city_type` sourcing is confirmed, copy the V3 notebook and modify:
- Audience creation query: add `city_type`
- Feature query: add GMV columns, remove promo filter
- Matching conditions: use `cfg.LOOK_ALIKE_CONDITIONS_V2`
- Write results with `technique='V2_CUSTOMER'`

---
## 1. Config & Authentication

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

# Mount Google Drive (one-time auth prompt)
from google.colab import drive
drive.mount('/content/drive')

# Point Python to the config directory on Drive
# Upload aa_config.py + campaign_data.csv to this folder ONCE
import sys, os
CONFIG_DIR = '/content/drive/MyDrive/aa_test'
sys.path.insert(0, CONFIG_DIR)
os.chdir(CONFIG_DIR)  # so aa_config can find campaign_data.csv

import aa_config as cfg

client = bigquery.Client(project=cfg.PROJECT_ID)
print(f'Connected to {cfg.PROJECT_ID}')
print(f'V2 matching conditions: {cfg.LOOK_ALIKE_CONDITIONS_V2}')

---
## 2. Discover city_type values

In [ ]:
# Run this to discover the city_type values used in V2 production
try:
    df_ct = client.query("""
        SELECT DISTINCT city_type, COUNT(*) AS cnt
        FROM `just-data-warehouse.customer_intelligence.customer_lookalike_evaluation_audience_citytype`
        GROUP BY 1 ORDER BY 2 DESC
    """).to_dataframe()
    print('city_type values in production V2 audience table:')
    print(df_ct.to_string(index=False))
except Exception as e:
    print(f'Could not query V2 audience table: {e}')

---
## 3. V2 Matching (Not Yet Implemented)

In [ ]:
raise NotImplementedError(
    'V2 Customer Lookalike A/A is not yet implemented.\n'
    'Resolve the city_type sourcing question above, then:\n'
    '  1. Copy the V3 notebook\n'
    '  2. Add city_type to the audience creation query\n'
    '  3. Modify the feature query (add GMV, remove promo filter)\n'
    '  4. Use cfg.LOOK_ALIKE_CONDITIONS_V2 for matching\n'
    '  5. Write results with technique=\'V2_CUSTOMER\''
)